In [13]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
!pip3 install tiktoken

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [14]:
import urllib.request
import ssl
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # Create an unverified SSL context
    ssl_context = ssl._create_unverified_context()

    # Downloading the file
    with urllib.request.urlopen(url, context=ssl_context) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # Unzipping the file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Add .tsv file extension
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)


sms_spam_collection/SMSSpamCollection.tsv already exists. Skipping download and extraction.


In [15]:
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df.head()

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [16]:
num_spam=df[df['Label']=='spam'].shape[0]
ham_subset=df[df['Label']=='ham'].sample(num_spam)
balanced_df=pd.concat([ham_subset,df[df['Label']=='spam']])

In [17]:
balanced_df["Label"]=balanced_df["Label"].map({'ham':0,'spam':1})

In [18]:
def random_split(df,train_ratio,val_ratio):
  df=df.sample(frac=1).reset_index(drop=True)
  train_end=int(len(df)*train_ratio)
  val_end=train_end + int(len(df)*val_ratio)
  train_data=df[:train_end]
  val_data=df[train_end:val_end]
  test_data=df[val_end:]
  return train_data,val_data,test_data
train_df,val_df,test_df=random_split(balanced_df,0.7,0.1)

In [19]:
train_df.to_csv("train.csv",index=None)
val_df.to_csv("val.csv",index=None)
test_df.to_csv("test.csv",index=None)

In [20]:
from torch.utils.data import Dataset,DataLoader

In [21]:
class CustomDataset(Dataset):
  def __init__(self,csv_path,tokenizer,max_length=None,eos_token_id=50256):
    self.data=pd.read_csv(csv_path)
    self.encoded_data=[tokenizer.encode(text) for text in self.data["Text"]]
    if max_length is None:
      self.max_length=self._longest_sequence_length()
    else:
      self.max_length=max_length
      self.encoded_data=[encoded_text[:self.max_length] for encoded_text in self.encoded_data]
    self.encoded_data=[encoded_text + [eos_token_id]*(self.max_length-len(encoded_text)) for encoded_text in self.encoded_data]

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
     label=self.data.iloc[index]["Label"]
     input=self.encoded_data[index]
     label_tensor=torch.tensor(label,dtype=torch.long)
     input_tensor=torch.tensor(input,dtype=torch.long)
     return input_tensor,label_tensor

  def _longest_sequence_length(self):
    max_length=0
    for text in self.encoded_data:
      max_length=max(max_length,len(text))
    return max_length


In [23]:
import tiktoken
tokenizer=tiktoken.get_encoding('gpt2')

In [24]:
train_dataset=CustomDataset(csv_path='/content/train.csv',tokenizer=tokenizer)

In [26]:
val_dataset=CustomDataset(csv_path='/content/val.csv',tokenizer=tokenizer,max_length=train_dataset.max_length)

In [27]:
test_dataset=CustomDataset(csv_path='/content/test.csv',tokenizer=tokenizer,max_length=train_dataset.max_length)

In [28]:
train_loader=DataLoader(train_dataset,batch_size=8,shuffle=True,drop_last=True)
val_loader=DataLoader(val_dataset,batch_size=8,shuffle=True,drop_last=True)
test_loader=DataLoader(test_dataset,batch_size=8,shuffle=True,drop_last=True)

In [32]:
from  gpt_download3 import download_and_load_gpt2

In [33]:
settings,params=download_and_load_gpt2(model_size="124M",models_dir="gpt2")

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 77.5kiB/s]
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 6.86MiB/s]
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verifica

In [41]:
from gpt_2_code_pretraining import GPTModel

In [42]:
config={"vocab_size":50257,"context_length":1024,"embed_dim":768,"num_heads":12,"n_layers":12,"dropout":0.0,"qkv_bias":True}

In [43]:
model=GPTModel(config)

In [44]:
from gpt_2_code_pretraining import load_weights_into_model

In [45]:
load_weights_into_model(model,params)

In [46]:
model.eval()

GPTModel(
  (token_embedding): Embedding(50257, 768)
  (pos_embedding): Embedding(1024, 768)
  (dropout): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiHeadAttention(
        (w_query): Linear(in_features=768, out_features=768, bias=True)
        (w_key): Linear(in_features=768, out_features=768, bias=True)
        (w_value): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (layer_norm1): LayerNorm()
      (layer_norm2): LayerNorm()
      (neural_net): FFNN(
        (network): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GeLU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (attn): MultiHeadAttention(
        (w_query): Linear(in_features=768, out_features=768, bias=True)
        (w_key): 

In [48]:
for param in model.parameters():
  param.requires_grad=False

In [49]:
num_classes=2
model.final_nn=nn.Linear(config["embed_dim"],num_classes,bias=False)

In [50]:
for param in model.trf_blocks[-1].parameters():
  param.requires_grad=True
for param in model.final_layer_norm.parameters():
  param.requires_grad=True

In [51]:
def calc_accuracy_total(data_loader,model,device,num_batches=None):
  model.eval()
  correct_predictions,num_examples=0,0
  if num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))
  for i,(input_batch,target_batch) in enumerate(data_loader):
    if i<num_batches:
      input_batch,target_batch=input_batch.to(device),target_batch.to(device)
      with torch.no_grad():
        output_batch=model(input_batch)[:,-1,:]
      predicted_labels=torch.argmax(output_batch,dim=-1)
      num_examples+=predicted_labels.shape[0]
      correct_predictions+=(predicted_labels==target_batch).sum().item()
    else:
      break
  return correct_predictions/num_examples

In [ ]:
def calc_loss_total(data_loader,model,device,num_batches=None):
  total_loss=0
  if len(data_loader)==0:
    return float("nan")
  elif num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))
  for i,(input_batch,target_batch) in enumerate(data_loader):
    if i<num_batches:
      input_batch,target_batch=input_batch.to(device),target_batch.to(device)
      output_batch=model(input_batch)
      loss=torch.nn.functional.cross_entropy(output_batch.flatten(0,1),target_batch.flatten())
      total_loss+=loss.item()
    else:
      break
  return total_loss/num_batches

In [ ]:
def train_model(model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq,eval_iter):
  global_step=-1
  train_losses=[]
  val_losses=[]
  for i in range(num_epochs):
    model.train()
    for i,(input_batch,target_batch) in enumerate(train_loader):
      input_batch,target_batch=input_batch.to(device),target_batch.to(device)
      output_batch=model(input_batch)
      loss=torch.nn.functional.cross_entropy(output_batch.flatten(0,1),target_batch.flatten())
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      global_step+=1
      if global_step%(eval_freq)==0:
        model.eval()
        with torch.no_grad():
            train_loss=calc_loss_total(train_loader,model=model,device=device,num_batches=eval_iter)
            val_loss=calc_loss_total(val_loader,model=model,device=device,num_batches=eval_iter)
            model.train()
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            print(f"Ep {i+1} (step {global_step:06d}):" f" Train loss:{train_loss:.3f}, Val loss:{val_loss:.3f}")
  return train_losses,val_losses




In [52]:
optimizer=torch.optim.AdamW(model.parameters(),lr=5e-5,weight_decay=0.1)

In [ ]:
num_epochs=5
train_losses,val_losses=train_model(model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq=50,eval_iter=5)